In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
import re

In [7]:
treadmill = pd.read_excel("../../data/cleaned/acceptances/treadmill.xlsx")
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")

In [3]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = treadmill.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,ARCADIA,ARKADIAN


In [8]:
unique_names = treadmill.horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [ ]:
import pandas as pd

# =========================
# 1. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    df['horse_name'] = df['horse_name'].astype(str).str.strip().str.upper()
    return df

runners = normalize(runners)
treadmill = normalize(treadmill)

# =========================
# 2. BUILD RUNNER MAP
# =========================
runner_map = runners[['meet_date', 'horse_name', 'race_no']].drop_duplicates()

runner_grouped = runner_map.groupby('meet_date')

# =========================
# 3. INITIAL MERGE
# =========================
merged = treadmill.merge(
    runner_map,
    on=['meet_date', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_old', '_true')
)

# =========================
# 4. SPLIT DATA
# =========================
exact_matched = merged[merged['_merge'] == 'both'].copy()
unmatched = merged[merged['_merge'] == 'left_only'].copy()

# =========================
# 5. PARTIAL MATCH (PREFIX)
# =========================
resolved_rows = []
still_unmatched = []

for _, row in unmatched.iterrows():
    date = row['meet_date']
    name = row['horse_name']
    
    if date not in runner_grouped.groups:
        still_unmatched.append(row)
        continue
    
    candidates = runner_grouped.get_group(date)
    
    matches = candidates[
        candidates['horse_name'].apply(lambda x: x.startswith(name))
    ]
    
    if len(matches) >= 1:
        best = matches.loc[matches['horse_name'].str.len().idxmax()]
        
        row['race_no'] = best['race_no']
        row['horse_name'] = best['horse_name']   # fix name
        
        resolved_rows.append(row)
    else:
        still_unmatched.append(row)

partial_matched = pd.DataFrame(resolved_rows)
edge_cases = pd.DataFrame(still_unmatched)

# =========================
# 6. CLEAN EXACT MATCH
# =========================
exact_matched['race_no'] = exact_matched['race_no_true']

exact_matched = exact_matched.drop(
    columns=['_merge', 'race_no_old', 'race_no_true']
)

# =========================
# 7. HANDLE NON-RUNNERS (KEEP THEM)
# =========================
if len(edge_cases) > 0:
    edge_cases['race_no'] = edge_cases['race_no_old']  # keep original
    edge_cases = edge_cases.drop(columns=['race_no_true', '_merge'])

# =========================
# 8. COMBINE ALL
# =========================
clean_treadmill = pd.concat(
    [exact_matched, partial_matched, edge_cases],
    ignore_index=True
)

# =========================
# 9. ADD FLAG
# =========================
clean_treadmill['is_current_runner'] = clean_treadmill['race_no'].notna()

# =========================
# 10. SORT + FORMAT
# =========================
clean_treadmill = clean_treadmill.sort_values(
    by=['meet_date', 'race_no']
).reset_index(drop=True)

clean_treadmill['meet_date'] = clean_treadmill['meet_date'].dt.strftime('%Y-%m-%d')
clean_treadmill['date'] = clean_treadmill['date'].dt.strftime('%Y-%m-%d')

# =========================
# 11. DEBUG
# =========================
print("Current runners:", clean_treadmill['is_current_runner'].sum())
print("Non-runners:", (~clean_treadmill['is_current_runner']).sum())

# =========================
# 12. SAVE
# =========================
clean_treadmill.to_excel(
    "../../data/cleaned/acceptances_cleaned/treadmill.xlsx",
    index=False
)

Current runners: 6473
Non-runners: 0


In [ ]:
# 1. Map out where the race number changes from the row above it
consecutive_groups = (bandages['race_no'] != bandages['race_no'].shift()).cumsum().values

# 2. Group by the array, the meet_date, and the race column
# This pulls meet_date into the resulting aggregation
race_counts = bandages.groupby([consecutive_groups, 'meet_date', 'race_no']).size().reset_index()

# 3. Rename columns cleanly to match the new structure
race_counts.columns = ['group_id', 'meet_date', 'race_no', 'count']

# 4. Filter for counts greater than 19 and display specific columns
filtered_counts = race_counts[race_counts['count'] > 19][['meet_date', 'race_no', 'count']]

with pd.option_context('display.max_rows', None):
    display(filtered_counts)

,meet_date,race_no,count
434,2011-02-06,184,22
437,2011-02-06,187,22
505,2011-03-06,255,22
668,2011-08-07,47,21
889,2011-11-27,34,20
1031,2012-02-04,176,22
1036,2012-02-04,181,20
1071,2012-02-19,216,20
1664,2013-02-02,171,20
1665,2013-02-02,172,20


In [ ]:
output_file = "../data/cleaned/acceptances_cleaned/acceptances.xlsx"

runners.to_excel(output_file, index=False)